# CTCSS and Signaling

This notebook focuses on the composite baseband in land-mobile FM: voice plus a low-frequency sub-audible tone used for selective squelch.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Voice Plus Tone

CTCSS tones live below the normal voice passband. They are present in the transmitter baseband, but ordinary speaker filtering tends to hide them from the listener.

In [ ]:
voice_bp = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
voice_bp = normalize(voice_bp)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_ctcss(tone_freq=141.3, tone_gain=0.25):
    tone = tone_gain * np.cos(2 * np.pi * tone_freq * t_work)
    composite = normalize(voice_bp + tone)
    speaker_hp = signal.sosfilt(signal.butter(5, 300, btype="highpass", fs=WORK_FS, output="sos"), composite)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(composite[:12000], fs=WORK_FS, ax=axes[0], title="Composite Baseband")
    plot_spectrum(composite, fs=WORK_FS, ax=axes[1], title="Composite Spectrum")
    axes[1].set_xlim(0, 4000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    with audio_out:
        audio_out.clear_output(wait=True)
        display(Markdown("**Composite audio (tone included)**"))
        display(audio_player(resample_signal(composite, WORK_FS, PLAY_FS), rate=PLAY_FS))
        display(Markdown("**Speaker-path audio (tone mostly removed)**"))
        display(audio_player(resample_signal(speaker_hp, WORK_FS, PLAY_FS), rate=PLAY_FS))

controls = widgets.interactive(
    update_ctcss,
    tone_freq=float_slider(min_value=67.0, max_value=254.1, step=0.1, value=141.3, description="Tone Hz"),
    tone_gain=float_slider(min_value=0.0, max_value=0.5, step=0.02, value=0.25, description="Tone gain"),
)
display(controls, audio_out)


## Key Takeaway

Selective signaling works because transmitters can carry information outside the normal voice band while receivers decide which parts of the composite signal to route to the speaker and which parts to route to control logic.